In [ ]:
import os
import json
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm import tqdm

import torch
from torch.utils.data import DataLoader

from core.models.MAE import MAEResNet, MAEResNetConfig
from core.datasets.pokemon_dataset import PokemonDataset
from core.models.utils import prepare_vae, get_number_of_parameters
from core.image_utils import show_batch_grid

In [ ]:
DEVICE = "cuda:0"
IMG_SIZE = 128

dataset_path = "./data/pokemon/images"
dataset_type = "POKEMON"

# dataset_path = "/home/msst/repo/drifting/data/CelebA"
# dataset_type = "FACES"


pokemon_dataset = PokemonDataset(root_dir=dataset_path, img_size=IMG_SIZE, dataset_type=dataset_type)

In [ ]:
vae_name = "flux-vae"

vae_kwargs = {
    "device_map": "cuda:0",
    "torch_dtype": torch.bfloat16,
}

vae, latent_shape = prepare_vae(
    vae_name="flux_vae",
    path_to_vae="./checkpoints/flux_vae",
    img_size=IMG_SIZE,
    vae_kwargs=vae_kwargs
)

print("VAE:")
get_number_of_parameters(vae)

# vae = torch.compile(vae, mode="reduce-overhead")

In [ ]:
mae_config_path = "./configs/MAE_configs/MAE-128_pixel_img128.json"
with open(mae_config_path, "r") as f:
    config_dict = json.load(f)

mae_config = MAEResNetConfig.from_dict(config_dict)
mae = MAEResNet(mae_config).cuda()

# mae = MAEResNet.from_pretrained(
#     ""
# ).cuda()

opt = torch.optim.AdamW(
    mae.parameters(), 
    lr=2e-4,
    betas=(0.9, 0.95),
    weight_decay=0.02,
)

In [ ]:
batch_size = 256
dataloader = DataLoader(pokemon_dataset, batch_size=batch_size, shuffle=True, num_workers=8)

In [ ]:
n_epochs = 250
plot_every = 100
save_step = 2500
save_path = f"/home/msst/repo/drifting/checkpoints/{dataset_type}/latent-MAE_{vae_name}_ch{mae.base_channels}_img{IMG_SIZE}_ch{mae.base_channels}_mask{mae.mask_ratio}_patch{mae.mask_patch_size}/"


steps_per_epoch = len(dataloader)

loss_history = []
plot_loss_acc = []
for e in range(n_epochs):
    for step, batch in enumerate(tqdm(dataloader)):
        global_step = e * steps_per_epoch + step + 1
        
        batch = batch.to(torch.bfloat16).cuda()
        with torch.no_grad():
            batch_latent = vae.encode(batch).latent_dist.sample()

        loss = mae_resnet(batch_latent, mask_ratio=MASK_RATIO)[0]
        loss.backward()
        # if step % gradient_accumulation == 0:
        opt.step()
        opt.zero_grad()
        
        plot_loss_acc.append(loss.item())
        
        
        if (global_step % plot_every == 0) or (global_step == 10):
            loss_history.append(sum(plot_loss_acc) / len(plot_loss_acc))
            plot_loss_acc = []
            clear_output(wait=True)
            plt.plot(loss_history)
            with torch.no_grad():
                test_imgs = next(iter(dataloader))[:8].to(torch.bfloat16).cuda()
                show_batch_grid(test_imgs, n=8, figsize=(9, 9), denormalize=True)

                test_imgs_latent = vae.encode(test_imgs).latent_dist.sample()
                mask = mae_resnet.make_patch_mask(test_imgs_latent, MASK_RATIO, 2)
                test_imgs_latent_masked = test_imgs_latent * (1.0 - mask)
                test_imgs_masked = vae.decode(test_imgs_latent_masked).sample
                show_batch_grid(test_imgs_masked, n=8, figsize=(9, 9), denormalize=True)

                feats_latent = mae_resnet.encoder(test_imgs_latent_masked)
                recon_latent = mae_resnet.decoder(feats_latent)
                recon = vae.decode(recon_latent).sample
                show_batch_grid(recon, n=8, figsize=(9, 9), denormalize=True)

                recon_latent = recon_latent * mask + test_imgs_latent_masked
                recon = vae.decode(recon_latent).sample
                show_batch_grid(recon, n=8, figsize=(9, 9), denormalize=True)
                                
                plt.show()
    
    if (e) % 25 == 0:
        os.makedirs(save_path, exist_ok=True)
        torch.save(mae_resnet.state_dict(), f"{save_path}/epochs{e}.pth")


In [ ]:
torch.save(mae_resnet.state_dict(), f"{save_path}/epochs{e}.pth")